In [1]:
import pandas as pd
import numpy as np
import ast
import re, string
import json
import csv
import bs4
from bs4 import BeautifulSoup
import requests
import time
import random
from tqdm import tqdm
import spacy
import gensim
# ! python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")



In [10]:
nonenglish_comms = r"""[^a-zA-Z\d\s\[\]\-\#\.\?\,\&\<\>\!\@\$\%\^\*\+\=\:\;\\/\%\'\"]"""

In [ ]:
alj_yt = pd.read_csv("D:\\hw\\macs-30122\\final-project-chattbd\\output_YT_AlJazeera.csv")
cnn_yt = pd.read_csv("D:\\hw\\macs-30122\\final-project-chattbd\\output_YT_CNN_approx252videos.csv")
fox_yt = pd.read_csv("D:\\hw\\macs-30122\\final-project-chattbd\\output_YT_FoxNews.csv")

### Helper funcs from other classes

In [2]:
# function from HW2 and HW4 of Content Analysis

def word_tokenize(word_list):
    """
    Take a list of words and tokenize the text
    Input: list of strs
    Returns a list of tokenized strs 
    """
    tokenized = []
    # pass word list through language model.
    doc = nlp(word_list)
    for token in doc:
        if not token.is_punct and len(token.text.strip()) > 0:
            tokenized.append(token.text)
    return tokenized


def normalizeTokens(word_list, extra_stop=[]):
    """
    Takes a list of words and normalizes the tokens
    Inputs:
        word_list: list of strs for words needed to be tokenized
        extra_stop: list of strs for extra stop words
    Returns list of tokenized strs
    """
    #We can use a generator here as we just need to iterate over it
    normalized = []
    if type(word_list) == list and len(word_list) == 1:
        word_list = word_list[0]

    if type(word_list) == list:
        word_list = ' '.join([str(elem) for elem in word_list])

    doc = nlp(word_list.lower())

    # add the property of stop word to words considered as stop words
    if len(extra_stop) > 0:
        for stopword in extra_stop:
            lexeme = nlp.vocab[stopword]
            lexeme.is_stop = True

    for w in doc:
        # if it's not a stop word or punctuation mark, add it to our article
        if w.text != '\n' and not w.is_stop and not w.is_punct \
            and not w.like_num and len(w.text.strip()) > 0:
            # we add the lematized version of the word
            normalized.append(str(w.lemma_))

    return normalized


def sent_tokenize(word_list, model=nlp):
    doc = model(word_list)
    sentences = [sent.text.strip() for sent in doc.sents]
    return sentences


# EDITED
# functions from TextClassification.ipynb MACS 30100

def decontracted(phrase):
    """
    Expand the contracted phrase into normal words
    """
    # specific
    phrase = re.sub(r"won't", "will not", phrase)
    phrase = re.sub(r"can\'t", "can not", phrase)
    # general
    phrase = re.sub(r"n\'t", " not", phrase)
    phrase = re.sub(r"\'re", " are", phrase)
    phrase = re.sub(r"\'s", " is", phrase) # prime 
    phrase = re.sub(r"\'d", " would", phrase)
    phrase = re.sub(r"\'ll", " will", phrase)
    phrase = re.sub(r"\'t", " not", phrase)
    phrase = re.sub(r"\'ve", " have", phrase)
    phrase = re.sub(r"\'m", " am", phrase)
    
    return phrase

# probably won't use
def clean_text(df):
    cleaned_post_text = []

    for post_text in tqdm(df):
        # remove extra white space
        post_text = re.sub(r"\s+", " ", post_text)
        # expand the contracted words
        post_text = decontracted(post_text)
        #remove html tags
        post_text = BeautifulSoup(post_text, 'lxml').get_text().strip() 
        post_text = re.sub(r'<.*?>', '', post_text)
        #remove url 
        post_text = re.sub(r'https*://\S+|www\.\S+', '', post_text)
        #Removing punctutation, string.punctuation in python consists of !"#$%&\'()*+,-./:;<=>?@[\\]^_{|}~`
        post_text = post_text.translate(str.maketrans('', '', string.punctuation))
        # ''.join([char for char in movie_text_data if char not in string.punctuation])
        # remove non word chrs
        post_text = re.sub("[^\w]"," ", post_text)
        # remove extra white space AGAIN 
        post_text = re.sub(r"\s+", " ", post_text)
        cleaned_post_text.append(post_text)

    return cleaned_post_text


def tag_sents_pos(sentences):
    """
    function which replicates NLTK pos tagging on sentences.
    """
    new_sents = []
    for sentence in sentences:
        new_sent = ' '.join(sentence)
        new_sents.append(new_sent)
    final_string = ' '.join(new_sents)
    doc = nlp(final_string)

    pos_sents = []
    for sent in doc.sents:
        pos_sent = []
        for token in sent:
            pos_sent.append((token.text, token.tag_))
        pos_sents.append(pos_sent)

    return pos_sents

### My helpers

In [4]:
def clean_transcript(transcript):
    """
    Takes a string that contains a transcript and removes extraneous noise,
    Including timestamps, CC information, etc, some other weird.
    Inputs:
        transcript: str
    Returns a cleaned transcript with text in lower case
    """

    # if auto generated in Arabic it would be much harder to perform our
    # task, so for simplicity we will remove such transcripts
    if re.search(r"Arabic \(auto-generated\)$", transcript):
        return []
    
    # speaker cites
    speaker = r">>\s*([\w]+):|(?<=\d{2}\s)(\w+)(?=\:)"
    # find the noise in the text, formatting things mostly
    noise = r"(>{1,2}|♪|\b(\d{1,2}\:)*\d{1,2}\:\d{2}\b)"
    # find the transcript formatting things that appear at the end
    cc = r"(?:English \(auto-generated\)|English|English \- cc1)\s*$"
    
    # remove artifacts of being closed captioning: speaker attr, timestamps, etc
    transcript = re.sub(speaker, "", transcript, flags=re.IGNORECASE)
    transcript = re.sub(noise, " ", transcript, flags=re.IGNORECASE)
    transcript = re.sub(cc, "", transcript, flags=re.IGNORECASE)
    # remove contractions
    transcript = decontracted(transcript)
    # remove the extra spaces, strip whitespace and lower case
    return re.sub(r"\s+", " ", transcript).strip().lower()


def clean_comm(comments, removers="", not_jank=True):
    """
    Cleans up the comment section, removing non-English symbols and URLs,
    among other things. For jankier comment sections removes more formatting also
    Input: 
        comments: a string that appears formatted as a list of strs
        removers: symbols to remove
        not_jank: bool to track whether the comments need more cleaning
    Returns a str combininb all cleaned comments
    """
    # remove contractions
    comm_str = decontracted(" ".join(ast.literal_eval(comments)))
    # remove html tags and URLs
    comm_str = BeautifulSoup(comm_str, 'lxml').get_text().strip() 
    comm_str = re.sub(r'<.*?>|https*://\S+|www\.\S+', '', comm_str)
    # remove problematic characters
    comm_str = re.sub(removers, "", comm_str)

    # should only be jank for CNN
    if not_jank:
        # return the comment as a 
        return comm_str.strip().lower()
    else:
        comment_clean = r"(?<=ago)\s*(?:\n|\(edited\)\n)(.*?)(?:Read more)*(?=\n(?:\d+\n)?Reply)"
        # I was having issues capturing all comments until I re.DOTALL
        # https://chat.openai.com/share/ed2187c5-dbcf-4a34-ac18-41e54890c812
        # consulted ChatGPT to help resolve this issue (using DOTALL)
        return " ".join(re.findall(comment_clean, comm_str, 
                                   flags=re.DOTALL)).strip().lower()


def clean_title(title, removers=""):
    """
    Cleans up YT titles
    Input: 
        title: a string corresponding to a video title
        removers: symbols to remove
    Returns cleaned title
    """
    title = decontracted(title)
    title = BeautifulSoup(title, 'lxml').get_text().strip() 
    title = re.sub(r'<.*?>|https*://\S+|www\.\S+', '', title)
    # remove problematic characters
    title = re.sub(removers, "", title)
    return title.strip().lower()


### Actual Data Cleaning

#### Youtube

##### RUN ONCE

In [5]:
def clean_yt(dfs,file_saves=["alj_yt.pkl", "cnn_yt.pkl", "fox_yt.pkl"]):
    """
    Take a list or iterable of pandas.DataFrames and edit them in place
    Essentially this function performs multiple tasks, it reformats columns,
    and cleans the data so it is more ready for textual analyses in the future
    Inputs:
        dfs: iterable of pandas.DataFrames
        file_saves: list of strs for filenames
    Returns None, edits DFs in place
    """
    # for progress bar
    tqdm.pandas()

    for i, df in enumerate(dfs):
        # change to datetime
        df["date"] = pd.to_datetime(df["date"])
        
        # replace all data that is uninformative with actually NaN values
        # https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.replace.html
        df.replace({'transcript': {'NA': np.nan,
                                "NaN": np.nan,
                                'Unable to obtain transcript': np.nan},
                    'comments': {'NA': np.nan, 
                                "NaN": np.nan,
                                "[]": np.nan}
                    }, inplace=True)
        
        # remove uninformative rows (no info in transcript or comments)
        df.drop(df[(df['transcript'].isna()) & 
            (df['comments'].isna())].index,inplace=True)

        # clean comments up
        nonenglish_comms = r"""[^a-zA-Z\d\s\[\]\-\#\.\?\,\&\<\>\!\@\$\%\^\*\+\=\:\;\\/\%\'\"]"""
        
        #the bool in clean_comm determines if more cleaning is necessary
        # based off whether the comment str starts with an "@"
        jank_start = re.compile(r"^((?:\[[\'\"])@)")
        df.comments = df.comments.apply(lambda x: 
                                        clean_comm(x, nonenglish_comms, 
                                        bool(not jank_start.match(x))) 
                                        if pd.notnull(x) else x)
        
        # clean transcript up, only run if not null
        df.transcript = df.transcript.apply(lambda x: clean_transcript(x)
                                            if pd.notnull(x) else x)
        # clean title
        df.title = df.title.apply(lambda x: clean_title(x, nonenglish_comms))
        # find useful text: a mix of title and transcript
        df["text"] = df.title + ". " + df.transcript.apply(lambda x: x if pd.notna(x) else "")
        # drop channel column (contains duplicate info), drop title, transcript
        df.drop(['channel',"title", "transcript"], axis=1, inplace=True)
        
        # tokenize and normalize comments
        df["toke_comms"] = df['comments'].progress_apply(lambda x: \
                           word_tokenize(x) if pd.notna(x) else x)
        df['word_counts_comms'] = df['toke_comms'].progress_apply(lambda x: \
                                  len(x) if isinstance(x, list) else x)
        df['norm_comms'] = df['toke_comms'].progress_apply(lambda x: \
                           normalizeTokens(x) if isinstance(x, list) else x)
        df['toke_comm_sents'] = df['comments'].progress_apply(lambda x: \
                                [word_tokenize(s) for s in sent_tokenize(x)] \
                                if pd.notna(x) else x)
        df['norm_comm_sents'] = df['toke_comm_sents'].progress_apply(lambda x: \
                                [normalizeTokens(s) for s in x] if isinstance(x, list) else x)
        
        # tokenize and normalize the texts
        df["toke_text"] = df['text'].progress_apply(lambda x: word_tokenize(x))
        df['word_counts'] = df['toke_text'].progress_apply(lambda x: len(x))
        df['norm_tokens'] = df['toke_text'].progress_apply(lambda x: \
                            normalizeTokens(x))
        df['toke_sents'] = df['text'].progress_apply(lambda x: \
                           [word_tokenize(s) for s in sent_tokenize(x)])
        df['norm_sents'] = df['toke_sents'].progress_apply(lambda x: \
                           [normalizeTokens(s) for s in x])

        # reset the index
        df.reset_index(drop=True, inplace=True)
        df.to_pickle(file_saves[i])
    return

In [ ]:
clean_yt([alj_yt, cnn_yt, fox_yt])

##### Load in Data (so not have to run above constantly)

In [7]:
alj_yt = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\alj_yt.pkl")
cnn_yt = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\cnn_yt.pkl")
fox_yt = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\fox_yt.pkl")

#### Reddit


##### RUN ONCE

In [ ]:
# read the pkl file
ip_df = pd.read_pickle("israel_palestine_df.pkl")

In [ ]:
# drop rows with no useful info
ip_df.dropna(subset=['post_text'], inplace=True)

In [ ]:
# clean up the post a little
ip_df.post_text = ip_df.post_text.apply(lambda x: clean_title(x, removers=nonenglish_comms))

C:\Users\Ethan\AppData\Local\Temp\ipykernel_2992\1870516207.py:72: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  title = BeautifulSoup(title, 'lxml').get_text().strip()
C:\Users\Ethan\AppData\Local\Temp\ipykernel_2992\1870516207.py:72: MarkupResemblesLocatorWarning: The input looks more like a URL than markup. You may want to use an HTTP client like requests to get the document behind the URL, and feed that document to Beautiful Soup.
  title = BeautifulSoup(title, 'lxml').get_text().strip()


In [ ]:
# normalize tokens and sents
tqdm.pandas()
ip_df["toke_text"] = ip_df['post_text'].progress_apply(lambda x: word_tokenize(x))
ip_df['word_counts'] = ip_df['toke_text'].progress_apply(lambda x: len(x))
ip_df['norm_tokens'] = ip_df['toke_text'].progress_apply(lambda x: normalizeTokens(x))
ip_df['toke_sents'] = ip_df['post_text'].progress_apply(lambda x: [word_tokenize(s) for s in sent_tokenize(x)])
ip_df['norm_sents'] = ip_df['toke_sents'].progress_apply(lambda x: [normalizeTokens(s) for s in x])

100%|██████████| 131719/131719 [55:13<00:00, 39.75it/s] 


In [ ]:
# save files
ip_df.to_pickle("ip_df_new.pkl")

##### Reopen file

In [ ]:
ip_df = pd.read_pickle("D:\\hw\\macs-30122\\project\\ip_df_new.pkl")

### News

#### Run once

In [52]:
fox_news = pd.read_pickle("D:\\hw\\macs-30122\\final-project-chattbd\\fox_articles.pkl")
# drop these useless columns
fox_news.drop(columns=['Unnamed: 0'], inplace=True)
# drop any possible duplicate rows based off of article url
fox_news.drop_duplicates(subset=['article_url'], inplace=True)
# drop rows with nas in all_text
fox_news.dropna(subset=["all_text"], inplace=True)
# reset indices
fox_news.reset_index(drop=True, inplace=True)
# add title to all_text
fox_news["text"] = fox_news.article_title + ". " + fox_news.all_text
# clean up the text a little
fox_news["text"] = fox_news.text.apply(lambda x: clean_title(x, nonenglish_comms))

In [53]:
# tokenization and normalization
tqdm.pandas()
fox_news["toke_text"] = fox_news['text'].progress_apply(lambda x: word_tokenize(x))
fox_news['word_counts'] = fox_news['toke_text'].progress_apply(lambda x: len(x))
fox_news['norm_tokens'] = fox_news['toke_text'].progress_apply(lambda x: \
                    normalizeTokens(x))
fox_news['toke_sents'] = fox_news['text'].progress_apply(lambda x: \
                    [word_tokenize(s) for s in sent_tokenize(x)])
fox_news['norm_sents'] = fox_news['toke_sents'].progress_apply(lambda x: \
                    [normalizeTokens(s) for s in x])


  0%|          | 0/1109 [00:00<?, ?it/s]

100%|██████████| 1109/1109 [05:39<00:00,  3.26it/s]


In [54]:
fox_news.to_pickle('fox_news.pkl')

#### open file

In [ ]:
fox_news = pd.read_pickle("fox_news.pkl")

### Word2Vec


In [ ]:
# combine all yt channels together
full_yt = pd.concat([alj_yt, cnn_yt, fox_yt])
# all yt comm sents
sm_yt = full_yt.norm_comm_sents[full_yt.comments.notna()].sum()
# all reddit sents
sm_red = ip_df.norm_sents.sum()

In [ ]:
# all social media sents for word2vec
full_sm = sm_yt + sm_red
with open("full_sm.csv", mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(full_sm)

##### Word2Vec model

In [ ]:
social_media_model = gensim.models.word2vec.Word2Vec(full_sm)
social_media_model.save("sm_word2vec.model")

In [57]:
news = fox_news
news_model = gensim.models.word2vec.Word2Vec(news)
news_model.save("news_word2vec.model")

#### Forgot to add POS tagger to all data

In [55]:
tqdm.pandas()
# full_yt['POS_sents'] = full_yt.toke_sents.progress_apply(lambda x: \
#     tag_sents_pos(x) if isinstance(x, list) else x)
fox_news["POS_sents"] = fox_news.toke_sents.progress_apply(lambda x: tag_sents_pos(x))
# ip_df["POS_sents"] = ip_df.toke_sents.progress_apply(lambda x: tag_sents_pos(x))

100%|██████████| 1109/1109 [02:15<00:00,  8.20it/s]


In [56]:
full_yt.to_pickle("full_yt.pkl")
fox_news.to_pickle('fox_news.pkl')
ip_df.to_pickle("ip_df_new.pkl")

In [59]:
fox_news

,article_url,article_date,article_title,all_text,text,toke_text,word_counts,norm_tokens,toke_sents,norm_sents,POS_sents
0,https://www.foxnews.com/politics/squad-democra...,2023-10-07,‘Squad’ Democrat calls for end of Israel’s ‘Ga...,"""We need a way to end this deadly violence tha...",squad democrat calls for end of israels gaza b...,"[squad, democrat, calls, for, end, of, israels...",299,"[squad, democrat, call, end, israels, gaza, bl...","[[squad, democrat, calls, for, end, of, israel...","[[squad, democrat, call, end, israels, gaza, b...","[[(squad, NNP), (democrat, NNP), (calls, VBZ),..."
1,https://www.foxnews.com/world/israels-military...,2023-10-07,Israel’s military says force is ‘at war’ with ...,"""There is no community in Southern Israel wher...",israels military says force is at war with ham...,"[israels, military, says, force, is, at, war, ...",629,"[israels, military, say, force, war, hamas, id...","[[israels, military, says, force, is, at, war,...","[[israels, military, say, force, war, hamas, i...","[[(israels, NNP), (military, NNP), (says, VBZ)..."
2,https://www.foxnews.com/politics/squad-dems-fa...,2023-10-07,‘Squad’ Dems face backlash calling for ‘ceasef...,Several members of the informal progressive ca...,squad dems face backlash calling for ceasefire...,"[squad, dems, face, backlash, calling, for, ce...",1322,"[squad, dem, face, backlash, call, ceasefire, ...","[[squad, dems, face, backlash, calling, for, c...","[[squad, dem, face, backlash, call, ceasefire,...","[[(squad, NNP), (dems, NNS), (face, VBP), (bac..."
3,https://www.foxnews.com/world/rocket-barrages-...,2023-10-07,At least 100 dead as Hamas launches unpreceden...,"A senior Hamas military commander, Mohammad De...",at least 100 dead as hamas launches unpreceden...,"[at, least, 100, dead, as, hamas, launches, un...",711,"[dead, hamas, launch, unprecedented, attack, i...","[[at, least, 100, dead, as, hamas, launches, u...","[[dead, hamas, launch, unprecedented, attack, ...","[[(at, RB), (least, RBS), (100, CD), (dead, JJ..."
4,https://www.foxnews.com/world/iran-funded-terr...,2023-10-07,Iran-funded terror proxies launch war against ...,"JERUSALEM, Israel —The Islamic Republic of Ira...",iran-funded terror proxies launch war against ...,"[iran, funded, terror, proxies, launch, war, a...",776,"[iran, funded, terror, proxy, launch, war, isr...","[[iran, funded, terror, proxies, launch, war, ...","[[iran, funded, terror, proxy, launch, war, is...","[[(iran, NNP), (funded, JJ), (terror, NN), (pr..."
...,...,...,...,...,...,...,...,...,...,...,...
1104,https://www.foxnews.com/media/washington-post-...,2023-12-29,Washington Post admits it ‘mischaracterized’ s...,A correction posted Dec. 28 clarified that it ...,washington post admits it mischaracterized sto...,"[washington, post, admits, it, mischaracterize...",361,"[washington, post, admit, mischaracterize, sto...","[[washington, post, admits, it, mischaracteriz...","[[washington, post, admit, mischaracterize, st...","[[(washington, NNP), (post, NNP), (admits, VBZ..."
1105,https://www.foxnews.com/us/pro-palestinian-pro...,2023-12-29,Pro-Palestinian protesters shout ‘Allahu akbar...,The group shut down the entrance to the World ...,pro-palestinian protesters shout allahu akbar ...,"[pro, palestinian, protesters, shout, allahu, ...",93,"[pro, palestinian, protester, shout, allahu, a...","[[pro, palestinian, protesters, shout, allahu,...","[[pro, palestinian, protester, shout, allahu, ...","[[(pro, JJ), (palestinian, JJ), (protesters, N..."
1106,https://www.foxnews.com/world/hamas-recent-bat...,2023-12-30,Hamas’ recent battle cry for violence against ...,JERUSALEM - The Hamas movement intensified its...,hamas recent battle cry for violence against u...,"[hamas, recent, battle, cry, for, violence, ag...",1017,"[hamas, recent, battle, cry, violence, late, l...","[[hamas, recent, battle, cry, for, violence, a...","[[hamas, recent, battle, cry, violence, late, ...","[[(hamas, NNP), (recent, JJ), (battle, NN), (c.